# FastAPI — Todo List CRUD

A REST API with full CRUD for a todo list, running inside Google Colab via `pyngrok`.

| Method | Endpoint | Action |
|--------|----------|--------|
| GET | `/todos` | List all todos |
| POST | `/todos` | Create a todo |
| GET | `/todos/{id}` | Get one todo |
| PUT | `/todos/{id}` | Update a todo |
| DELETE | `/todos/{id}` | Delete a todo |

In [1]:
# Install dependencies
!pip install fastapi uvicorn pyngrok nest_asyncio --quiet


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading

nest_asyncio.apply()

In [3]:
# ── Models ──────────────────────────────────────────────────────────────────

class TodoCreate(BaseModel):
    title: str
    description: Optional[str] = None
    completed: bool = False

class TodoUpdate(BaseModel):
    title: Optional[str] = None
    description: Optional[str] = None
    completed: Optional[bool] = None

class Todo(TodoCreate):
    id: int


# ── App & in-memory store ────────────────────────────────────────────────────

app = FastAPI(title="Todo API")

db: dict[int, Todo] = {}
next_id = 1


# ── Routes ──────────────────────────────────────────────────────────────────

@app.get("/todos", response_model=list[Todo])
def list_todos():
    return list(db.values())


@app.post("/todos", response_model=Todo, status_code=201)
def create_todo(payload: TodoCreate):
    global next_id
    todo = Todo(id=next_id, **payload.model_dump())
    db[next_id] = todo
    next_id += 1
    return todo


@app.get("/todos/{todo_id}", response_model=Todo)
def get_todo(todo_id: int):
    if todo_id not in db:
        raise HTTPException(status_code=404, detail="Todo not found")
    return db[todo_id]


@app.put("/todos/{todo_id}", response_model=Todo)
def update_todo(todo_id: int, payload: TodoUpdate):
    if todo_id not in db:
        raise HTTPException(status_code=404, detail="Todo not found")
    todo = db[todo_id]
    updates = payload.model_dump(exclude_unset=True)
    db[todo_id] = todo.model_copy(update=updates)
    return db[todo_id]


@app.delete("/todos/{todo_id}", status_code=204)
def delete_todo(todo_id: int):
    if todo_id not in db:
        raise HTTPException(status_code=404, detail="Todo not found")
    del db[todo_id]

In [4]:
# ── Start server ─────────────────────────────────────────────────────────────
# Add your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("YOUR_TOKEN_HERE")

PORT = 8000

public_url = ngrok.connect(PORT)
print(f"Public URL : {public_url}")
print(f"Swagger UI : {public_url}/docs")

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

t=2026-04-09T13:37:18-0400 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
t=2026-04-09T13:37:18-0400 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
t=2026-04-09T13:37:18-0400 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
# ── Test the API with the requests library ────────────────────────────────────
import requests

BASE = f"http://localhost:{PORT}"

# CREATE
r = requests.post(f"{BASE}/todos", json={"title": "Buy groceries", "description": "Milk, eggs, bread"})
print("CREATE:", r.status_code, r.json())

r = requests.post(f"{BASE}/todos", json={"title": "Walk the dog"})
print("CREATE:", r.status_code, r.json())

# READ ALL
r = requests.get(f"{BASE}/todos")
print("\nLIST:", r.status_code, r.json())

# READ ONE
r = requests.get(f"{BASE}/todos/1")
print("\nGET 1:", r.status_code, r.json())

# UPDATE
r = requests.put(f"{BASE}/todos/1", json={"completed": True})
print("\nUPDATE 1:", r.status_code, r.json())

# DELETE
r = requests.delete(f"{BASE}/todos/2")
print("\nDELETE 2:", r.status_code)

# FINAL STATE
r = requests.get(f"{BASE}/todos")
print("\nFINAL LIST:", r.json())